# LEGO VR Analysis - All Participants

Load all eye-tracking CSV files for every participant and combine them into one dataframe.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

DATA_DIR = Path("../data/eye_tracking")
CONDITIONS = [1, 2, 3]

csv_files = sorted(DATA_DIR.glob("*_ET_Data_Condition*_*.csv"))

participant_ids = sorted({file_path.name.split("_")[0] for file_path in csv_files})

dfs = []
missing_files = {}

for participant_id in participant_ids:
    for condition in CONDITIONS:
        matches = sorted(DATA_DIR.glob(f"{participant_id}_ET_Data_Condition{condition}_*.csv"))

        if not matches:
            missing_files.setdefault(participant_id, []).append(condition)
            continue

        file_path = matches[0]

        df_condition = pd.read_csv(file_path, low_memory=False)
        df_condition["participant_id"] = participant_id
        df_condition["condition_number"] = condition
        df_condition["source_file"] = file_path.name

        dfs.append(df_condition)

if not dfs:
    raise FileNotFoundError(f"No condition files found in {DATA_DIR}")

df = pd.concat(dfs, ignore_index=True)

print(f"Loaded {len(dfs)} files from {len(df['participant_id'].unique())} participants.")

if missing_files:
    print("Missing condition files:")
    for participant_id, conditions in missing_files.items():
        print(f"  Participant {participant_id}: missing conditions {conditions}")

df.head()